# 🎯 09 — Fine-tune SIM→REAL (Fase 3): candidato experimental

> **Célula única — `Runtime → Run all`.** Use GPU. Exige `HF_REPO_ID` ASCII no formato `owner/repo` e `HF_BASELINE_REVISION`; publicação de candidato exige também `POKER_PUBLISH_ARTIFACTS=1` e `HF_TOKEN`.

## Por que
O F2 foi avaliado apenas nos estilos do próprio gerador. Telas reais têm anti-aliasing, sombras, compressão, fontes e ícones ausentes do sintético. Este notebook testa a hipótese de fine-tune a partir de um checkpoint pinado; não promete uma quantidade universal de imagens nem generalização real.

## Controles contra regressão
1. **Parte do `best.pt` do F2**, não do zero — preserva tudo que ele já aprendeu.
2. **Congela o backbone** (`freeze=10`): só a cabeça de detecção se adapta ao domínio real (poucos dados não destroem as features).
3. **Mistura sintético + real** no treino: senão o modelo *esquece* as 52 cartas que suas poucas telas reais não cobrem todas (esquecimento catastrófico).
4. **Valida em REAL held-out separado POR FONTE**: telas do mesmo cliente/sessão nunca aparecem em treino **e** validação ao mesmo tempo — senão o número mente (vazamento de telas quase-iguais).

## Como rotular telas reais barato (human-in-the-loop)
A função `prelabel` pode criar rótulos candidatos, mas todos precisam de revisão humana cega. Não há medição local que sustente um fator fixo de economia de tempo.

## Como usar
- **Com telas reais:** use pares imagem/rótulo com prefixo de grupo e pelo menos quatro grupos independentes. O notebook exporta um candidato; nunca promove o canônico.
- **Sem telas reais:** roda somente um stress-test sintético de integração. Esse resultado não autoriza publicação nem alegação de ganho real.

## Autossuficiente (checkpoint base)
O passo 0 baixa `table_yolo11n.pt` de uma revisão HF imutável. Ausência ou divergência do baseline bloqueia o experimento; não existe fallback silencioso para outro treino.

## Depois
O artefato permanece em `candidates/<sha256>/`. A promoção requer o benchmark independente descrito em `docs/research/STATE_OF_ART_20260717.md`, incluindo exact-state, FAR, calibração, latência e paridade ONNX.

*Classes idênticas às do 08 (52 cartas + `seat` + `button`), então o decode do backend e a geometria de posição continuam iguais.*


In [ ]:
# ============================================================
# FINE-TUNE SIM->REAL (Fase 3) - candidato experimental
# Celula UNICA. Baseline pinado; publicacao somente de candidato com opt-in.
#
# POR QUE: o F2 foi avaliado apenas nos estilos do gerador sintetico,
# mas telas REAIS de clientes tem detalhes que o sintetico nao cobre (anti-aliasing, sombras,
# compressao, fontes proprietarias). A alavanca comprovada pra fechar o gap sim->real e um
# e esta etapa testa fine-tune com telas reais rotuladas e agrupadas.
#
# RIGOR (as 4 regras que fazem o fine-tune nao piorar o modelo):
#  1) parte do best.pt do F2 (nao do zero);  2) CONGELA o backbone (so ajusta a cabeca);
#  3) MISTURA sintetico + real (senao esquece as 52 cartas que o real nao cobre todas);
#  4) VALIDA em REAL held-out separado POR FONTE (sem vazamento de telas quase-iguais).
# Sem telas reais, o stress-test sintetico valida apenas a integracao.
# ============================================================
import glob
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
from pathlib import Path

ULTRALYTICS_VERSION = '8.3.223'
HUGGINGFACE_HUB_VERSION = '0.34.3'
PILLOW_VERSION = '12.3.0'
NUMPY_VERSION = '2.3.1'
safe_env_names = ('PATH', 'HOME', 'LANG', 'LC_ALL', 'TMPDIR', 'SYSTEMROOT', 'WINDIR', 'USERPROFILE', 'TEMP', 'TMP', 'SSL_CERT_FILE', 'SSL_CERT_DIR')
tool_env = {name: os.environ[name] for name in safe_env_names if name in os.environ}
pip_env = {**tool_env, 'PIP_INDEX_URL': 'https://pypi.org/simple', 'PIP_CONFIG_FILE': os.devnull}
subprocess.run(  # noqa: S603 - sys.executable e versoes exatas
    [sys.executable, '-m', 'pip', 'install', '-q',
     f'ultralytics=={ULTRALYTICS_VERSION}',
     f'huggingface_hub=={HUGGINGFACE_HUB_VERSION}',
     f'pillow=={PILLOW_VERSION}', f'numpy=={NUMPY_VERSION}'],
    check=True, env=pip_env,
)
del pip_env

import numpy as np
import PIL
import torch
import ultralytics
from PIL import Image, ImageDraw, ImageFilter, ImageFont


def box(t, ls=()):
    w = max([len(t)] + [len(x) for x in ls]) if ls else len(t)
    print("\n  +" + "-" * (w + 2) + "+")
    print("  | " + t.ljust(w) + " |")
    print("  +" + "-" * (w + 2) + "+")
    for x in ls:
        print("  | " + x.ljust(w) + " |")
    if ls:
        print("  +" + "-" * (w + 2) + "+")


# ---------- CONFIG ----------
ROOT = Path("/content/ft_run_20260717")
BASE_PT_NAME = "table_yolo11n.pt"    # checkpoint do F2 (o notebook 08 publica isso)
REAL_DIR = "/content/real"           # SUAS telas: real/images/*.png (+ real/labels/*.txt YOLO)
REAL_MANIFEST_PATH = Path(REAL_DIR) / 'manifest.json'
PROJECT_ROOT = Path(os.environ.get('POKER_PROJECT_ROOT', '/content/poker-arena'))
PROJECT_COMMIT = os.environ.get('POKER_PROJECT_COMMIT', '')
EPOCHS_FT, IMGSZ = 40, 640
FREEZE = 10                          # congela o backbone (so a cabeca aprende o dominio real)
LR0_FT = 0.001                       # 1/10 do treino do zero (nao destroi o que o F2 aprendeu)
SYNTH_MIX = 1200                     # sinteticas no treino (mantem as 54 classes vivas)
SEED = 20260717
PUBLISH_ARTIFACTS = os.environ.get("POKER_PUBLISH_ARTIFACTS") == "1"
RANKS, SUITS = "23456789TJQKA", "shdc"
CLASSES = [r + s for r in RANKS for s in SUITS]
SEAT, BUTTON = 52, 53
NAMES = CLASSES + ["seat", "button"]
NC = len(NAMES)
RANK_DISP = {"T": "10"}
if re.fullmatch(r'[0-9a-f]{40}', PROJECT_COMMIT) is None:
    raise RuntimeError('POKER_PROJECT_COMMIT deve ser SHA lowercase de 40 hex.')
git_executable = shutil.which('git')
if not git_executable:
    raise RuntimeError('git nao esta instalado no runtime.')
try:
    resolved_project_commit = subprocess.check_output(  # noqa: S603 - git resolvido e comando fixo
        [git_executable, '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True, env=tool_env,
    ).strip()
except (OSError, subprocess.CalledProcessError) as exc:
    raise RuntimeError('checkout local pinado do projeto e obrigatorio.') from exc
if resolved_project_commit != PROJECT_COMMIT:
    raise RuntimeError('checkout local diverge de POKER_PROJECT_COMMIT.')
working_tree = subprocess.check_output(  # noqa: S603 - git resolvido e checkout do operador
    [git_executable, '-C', str(PROJECT_ROOT), 'status', '--porcelain', '--untracked-files=all'],
    text=True, env=tool_env,
).strip()
if working_tree:
    raise RuntimeError('checkout local contem alteracoes ou arquivos nao rastreados.')
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
from poker_arena.ml.data_manifest import validate_manifest
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

box("FINE-TUNE SIM->REAL (F3) - o plano", [
    "0. obter o checkpoint do F2 (best.pt): baixa do HF, OU treina um base agora",
    "1. PRE-ROTULAR suas telas reais com o F2 (voce so CORRIGE - rotulo barato)",
    "2. montar dataset: real (train) + sintetico (mistura) ; REAL held-out por FONTE",
    "3. fine-tune: backbone CONGELADO, LR baixo, poucas epocas",
    "4. medir BASE vs AJUSTADO no REAL held-out -> so publica se MELHOROU",
])

get_ipython().system('apt-get -qq install -y fonts-liberation fonts-dejavu-core fonts-freefont-ttf >/dev/null 2>&1')
FONTS = glob.glob("/usr/share/fonts/truetype/**/*.ttf", recursive=True)
FONTS = [f for f in FONTS if not any(b in f.lower() for b in ("emoji", "symbol", "mono"))][:14] or [None]


# ---------- gerador sintetico (mesma ideia do 08: cartas+jogadores+botao) ----------
def draw_suit(d, suit, cx, cy, s, color):
    if suit == "d":
        d.polygon([(cx, cy - s), (cx + s * .7, cy), (cx, cy + s), (cx - s * .7, cy)], fill=color)
    elif suit == "h":
        d.pieslice([cx - s, cy - s, cx, cy], 0, 360, fill=color)
        d.pieslice([cx, cy - s, cx + s, cy], 0, 360, fill=color)
        d.polygon([(cx - s * .98, cy - s * .1), (cx + s * .98, cy - s * .1), (cx, cy + s)], fill=color)
    elif suit == "s":
        d.polygon([(cx, cy - s), (cx + s * .85, cy + s * .4), (cx - s * .85, cy + s * .4)], fill=color)
        d.pieslice([cx - s * .85, cy - s * .1, cx, cy + s * .6], 0, 360, fill=color)
        d.pieslice([cx, cy - s * .1, cx + s * .85, cy + s * .6], 0, 360, fill=color)
        d.rectangle([cx - s * .14, cy + s * .2, cx + s * .14, cy + s], fill=color)
    else:
        r = s * .5
        d.ellipse([cx - r, cy - s * .9, cx + r, cy + r * .2], fill=color)
        d.ellipse([cx - s * .9, cy - r * .3, cx - s * .9 + 2 * r, cy - r * .3 + 2 * r], fill=color)
        d.ellipse([cx + s * .9 - 2 * r, cy - r * .3, cx + s * .9, cy - r * .3 + 2 * r], fill=color)
        d.rectangle([cx - s * .14, cy, cx + s * .14, cy + s], fill=color)


def render_card(rng, rank, suit, w):
    h = int(w * 1.4)
    img = Image.new("RGB", (w, h), (0, 0, 0))
    bg = tuple(rng.randint(238, 255) for _ in range(3))
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([0, 0, w - 1, h - 1], radius=rng.randint(3, max(4, w // 8)), fill=bg,
                        outline=tuple(rng.randint(150, 200) for _ in range(3)), width=rng.randint(1, 2))
    four = rng.random() < 0.4
    cmap = {"s": (25, 25, 25), "c": ((20, 120, 40) if four else (25, 25, 25)),
            "h": (200, 30, 40), "d": ((30, 90, 210) if four else (200, 30, 40))}
    col = tuple(int(np.clip(c + rng.randint(-25, 25), 0, 255)) for c in cmap[suit])
    try:
        f = ImageFont.truetype(rng.choice(FONTS), int(w * rng.uniform(0.30, 0.40)))
    except Exception:
        f = ImageFont.load_default()
    d.text((w * 0.08, h * 0.03), RANK_DISP.get(rank, rank), font=f, fill=col)
    draw_suit(d, suit, int(w * 0.24), int(h * 0.30), w * 0.11, col)
    draw_suit(d, suit, w // 2, int(h * 0.62), w * rng.uniform(0.22, 0.28), col)
    return img


def gen_sample(rng, sseed, hard=False):
    """Mesa 2D + labels YOLO (54 classes). hard=True = dominio 'mais real/dificil'
    (compressao, cast de cor, leve blur) usado no stress-test quando falta tela real."""
    W, H = 900, 600
    srng = random.Random(sseed)
    felt = tuple(srng.randint(15, 110) for _ in range(3))
    img = Image.new("RGB", (W, H), felt)
    d = ImageDraw.Draw(img)
    if srng.random() < 0.7:
        d.ellipse([W * .08, H * .03, W * .92, H * .97],
                  fill=tuple(min(255, c + srng.randint(8, 30)) for c in felt))
    labels = []
    n = rng.randint(2, 9)
    cx0, cy0, rx, ry = W / 2, H / 2, W * 0.40, H * 0.44
    seats = [(cx0 + rx * math.cos(math.radians(90 + i * 360 / n)),
              cy0 + ry * math.sin(math.radians(90 + i * 360 / n))) for i in range(n)]
    for (sx, sy) in seats:
        rr = srng.randint(20, 34)
        acol = tuple(srng.randint(20, 235) for _ in range(3))
        d.ellipse([sx - rr, sy - rr, sx + rr, sy + rr], fill=acol, outline=(235, 235, 235), width=2)
        labels.append((SEAT, sx / W, sy / H, (2 * rr) / W, (2 * rr) / H))
    deck = CLASSES[:]
    rng.shuffle(deck)
    nb = rng.choice([0, 3, 3, 4, 5])
    hole, board = deck[:2], deck[2:2 + nb]
    cw = int(srng.uniform(40, 84))

    def place(card, cx, cy):
        ci = render_card(rng, card[0], card[1], cw)
        img.paste(ci, (int(cx - ci.width / 2), int(cy - ci.height / 2)))
        labels.append((CLASSES.index(card), cx / W, cy / H, ci.width / W, ci.height / H))
    if board:
        gap = int(cw * 1.3)
        x0 = W // 2 - (len(board) - 1) * gap // 2
        by = H // 2 + rng.randint(-25, 10)
        for i, c in enumerate(board):
            place(c, x0 + i * gap + rng.randint(-4, 4), by)
    hx, hy = W // 2 + rng.randint(-50, 50), int(H * 0.76) + rng.randint(-12, 10)
    for i, c in enumerate(hole):
        place(c, hx - 46 + i * 92, hy)
    bseat = rng.randrange(n)
    sxb, syb = seats[bseat]
    ux, uy = (cx0 - sxb), (cy0 - syb)
    lg = math.hypot(ux, uy) or 1.0
    ux, uy = ux / lg, uy / lg
    bx, by2 = sxb + ux * 16 - uy * 42, syb + uy * 16 + ux * 42
    brr = srng.randint(10, 16)
    d.ellipse([bx - brr, by2 - brr, bx + brr, by2 + brr], fill=(232, 200, 60), outline=(120, 95, 10), width=2)
    labels.append((BUTTON, bx / W, by2 / H, (2 * brr) / W, (2 * brr) / H))
    if hard:  # 'mais real': cast de cor + blur + compressao forte
        img = Image.blend(img, Image.new("RGB", (W, H), (srng.randint(0, 60), srng.randint(0, 40), srng.randint(0, 60))), 0.12)
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.4, 1.3)))
    elif rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0, 0.8)))
    return img, labels


def write_split(split, imgs_labels):
    os.makedirs(f"{ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
    for k, (img, labels) in enumerate(imgs_labels):
        q = random.Random(k).randint(35, 70)  # compressao variavel (mais 'real')
        img.save(f"{ROOT}/images/{split}/{split}_{k:05}.jpg", quality=q)
        with open(f"{ROOT}/labels/{split}/{split}_{k:05}.txt", "w") as fh:
            fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c, x, y, w, h in labels))


# ---------- STEP 0: baixar checkpoint base de revisao imutavel ----------
box("[0/4] Obtendo checkpoint base pinado (best.pt)")
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
try:
    from google.colab import userdata
except ImportError:
    userdata = None
baseline_auth_value = (userdata.get("HF_TOKEN") if userdata is not None
            else os.environ.get("HF_TOKEN"))
os.environ.pop('HF_TOKEN', None)
HF_BASELINE_REVISION = (userdata.get("HF_BASELINE_REVISION") if userdata is not None
                        else os.environ.get("HF_BASELINE_REVISION"))
HF_REPO_ID = (userdata.get("HF_REPO_ID") if userdata is not None
              else os.environ.get("HF_REPO_ID"))
sensitive_env_name = re.compile(r'(?i)(^|_)(TOKEN|SECRET|PASSWORD|PASSWD|PRIVATE_KEY|CREDENTIALS?|API_KEY|ACCESS_KEY)($|_)')
for inherited_name in tuple(os.environ):
    if sensitive_env_name.search(inherited_name):
        os.environ.pop(inherited_name, None)
if not baseline_auth_value:
    raise RuntimeError("HF_TOKEN e obrigatorio para o baseline privado")
HF_REPO_ID_PATTERN = r'[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?/[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?'
if (not isinstance(HF_REPO_ID, str) or not HF_REPO_ID.isascii()
        or len(HF_REPO_ID) > 96 or re.fullmatch(HF_REPO_ID_PATTERN, HF_REPO_ID) is None
        or '..' in HF_REPO_ID or '--' in HF_REPO_ID):
    raise RuntimeError('HF_REPO_ID deve ser ASCII owner/repo canonico.')
if (not isinstance(HF_BASELINE_REVISION, str)
        or re.fullmatch(r'[0-9a-f]{40}', HF_BASELINE_REVISION) is None):
    raise RuntimeError("HF_BASELINE_REVISION deve ser commit SHA lowercase imutavel")


base_pt = hf_hub_download(
    HF_REPO_ID, BASE_PT_NAME, revision=HF_BASELINE_REVISION,
    token=baseline_auth_value, local_dir=str(ROOT),
)
print("checkpoint base pinado:", HF_BASELINE_REVISION)
base = YOLO(base_pt)
baseline_auth_value = ''

# ---------- STEP 1: telas REAIS (ou stress-test de dominio) ----------
real_imgs = sorted(glob.glob(f"{REAL_DIR}/images/*.*"))
real_labels = sorted(glob.glob(f"{REAL_DIR}/labels/*.txt"))
real_image_stems = {Path(p).stem for p in real_imgs}
real_label_stems = {Path(p).stem for p in real_labels}
if real_image_stems != real_label_stems:
    missing_labels = sorted(real_image_stems - real_label_stems)
    missing_images = sorted(real_label_stems - real_image_stems)
    raise RuntimeError(
        f"pares imagem/label divergentes: sem label={missing_labels[:5]}, "
        f"sem imagem={missing_images[:5]}"
    )
if real_imgs and len(real_imgs) < 20:
    raise RuntimeError("dataset real parcial: minimo operacional de 20 pares")
have_real = len(real_imgs) >= 20


def validate_yolo_label(path):
    rows = path.read_text(encoding='utf-8').splitlines()
    if not rows:
        raise RuntimeError('label YOLO vazio.')
    for line in rows:
        parts = line.split()
        if len(parts) != 5:
            raise RuntimeError('label YOLO deve ter cinco campos.')
        try:
            cls = int(parts[0])
            cx, cy, width, height = map(float, parts[1:])
        except ValueError as exc:
            raise RuntimeError('label YOLO contem numero invalido.') from exc
        if (not 0 <= cls < NC or not np.isfinite([cx, cy, width, height]).all()
                or not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < width <= 1 and 0 < height <= 1)
                or cx - width / 2 < 0 or cx + width / 2 > 1
                or cy - height / 2 < 0 or cy + height / 2 > 1):
            raise RuntimeError('label YOLO fora do contrato de classe/coordenadas.')


def difference_hash(path):
    with Image.open(path) as image:
        pixels = np.asarray(image.convert('L').resize((17, 16), Image.Resampling.LANCZOS))
    bits = (pixels[:, 1:] > pixels[:, :-1]).reshape(-1)
    return sum(int(bit) << index for index, bit in enumerate(bits))

if have_real:
    governance_report = validate_manifest(
        REAL_MANIFEST_PATH, REAL_DIR, requested_use='model-training',
    )
    if not governance_report.valid or governance_report.manifest_receipt is None:
        codes = sorted({issue.code for issue in governance_report.errors})
        raise RuntimeError(f'manifesto de dados reais reprovado: {codes}')
    manifest_payload = json.loads(REAL_MANIFEST_PATH.read_text(encoding='utf-8'))
    samples = manifest_payload['samples']
    sample_by_stem = {Path(sample['path']).stem: sample for sample in samples}
    if len(sample_by_stem) != len(samples) or set(sample_by_stem) != real_image_stems:
        raise RuntimeError('manifesto deve identificar exatamente cada imagem real uma vez.')
    if any(sample['split'] not in {'train', 'validation'} for sample in samples):
        raise RuntimeError('fine-tune aceita somente splits train/validation governados.')
    for stem in sorted(real_image_stems):
        validate_yolo_label(Path(REAL_DIR) / 'labels' / f'{stem}.txt')
    box("[1/4] Telas REAIS encontradas", [f"{len(real_imgs)} imagens + labels em {REAL_DIR}"])
    # Split e grupos vem do manifesto validado, nunca do nome do arquivo.
    def sample_for(path):
        return sample_by_stem[Path(path).stem]
    srcs = sorted({sample['split_group'] for sample in samples})
    if len(srcs) < 4:
        raise RuntimeError("sao exigidos ao menos 4 grupos/fontes independentes")
    group_sizes = {s: sum(sample['split_group'] == s for sample in samples) for s in srcs}
    if min(group_sizes.values()) < 5:
        raise RuntimeError(f"cada grupo precisa de >=5 imagens: {group_sizes}")
    tr = [p for p in real_imgs if sample_for(p)['split'] == 'train']
    va = [p for p in real_imgs if sample_for(p)['split'] == 'validation']
    if not tr or not va:
        raise RuntimeError("split agrupado produziu treino ou validacao vazios")
    tr_hashes = [(p, difference_hash(p)) for p in tr]
    va_hashes = [(p, difference_hash(p)) for p in va]
    if any((left_hash ^ right_hash).bit_count() <= 4
           for _left, left_hash in tr_hashes for _right, right_hash in va_hashes):
        raise RuntimeError('pHash detectou duplicata/quase-duplicata entre train e validation.')
    split_inventory = []
    for p in [*tr, *va]:
        sample = sample_for(p)
        label_path = Path(REAL_DIR) / 'labels' / f'{Path(p).stem}.txt'
        split_inventory.append({
            'id': sample['id'], 'split': sample['split'],
            'split_group': sample['split_group'], 'image_sha256': sample['sha256'],
            'label_sha256': hashlib.sha256(label_path.read_bytes()).hexdigest(),
            'phash': f'{difference_hash(p):064x}',
        })
    split_inventory.sort(key=lambda item: item['id'])
    split_assignment_sha256 = hashlib.sha256(
        json.dumps(split_inventory, sort_keys=True, separators=(',', ':')).encode(),
    ).hexdigest()
    real_data_receipt = {
        'schema_version': 1, 'governance': governance_report.as_dict(),
        'split_assignment_sha256': split_assignment_sha256,
        'sample_count': len(split_inventory), 'phash_cross_split_max_distance': 4,
        'inventory': split_inventory,
    }
    print(f"grupos: {len(srcs)} | treino {len(tr)} imgs | held-out {len(va)} imgs")
    for split, ps in (("train", tr), ("val", va)):
        os.makedirs(f"{ROOT}/images/{split}", exist_ok=True)
        os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
        for p in ps:
            shutil.copy(p, f"{ROOT}/images/{split}/{os.path.basename(p)}")
            lp = f"{REAL_DIR}/labels/{os.path.splitext(os.path.basename(p))[0]}.txt"
            shutil.copy(lp, f"{ROOT}/labels/{split}/{os.path.splitext(os.path.basename(p))[0]}.txt")
    DEMO = False
else:
    box("[1/4] Sem telas reais -> STRESS-TEST de dominio", [
        "gero um dominio sintetico MAIS DURO (cast/blur/compressao) como 'pseudo-real'",
        "prova a maquinaria e MEDE o ganho; troque por telas reais em " + REAL_DIR,
        "PRE-ROTULO: rode base.predict nas suas telas p/ rotulo barato (veja o codigo)",
    ])
    rng = random.Random(2024)
    write_split("train", [gen_sample(rng, rng.randint(200, 260), hard=True) for _ in range(160)])
    write_split("val", [gen_sample(rng, rng.randint(260, 300), hard=True) for _ in range(80)])
    real_data_receipt = {
        'schema_version': 1, 'governance': None, 'sample_count': 0,
        'split_assignment_sha256': None, 'demo_only': True,
    }
    DEMO = True

# mistura sintetica 'facil' no treino (anti-esquecimento das 54 classes)
box("[2/4] Misturando sintetico ao treino (anti-esquecimento)")
rngm = random.Random(7)
os.makedirs(f"{ROOT}/images/train", exist_ok=True)
os.makedirs(f"{ROOT}/labels/train", exist_ok=True)
for k in range(SYNTH_MIX):
    img, labels = gen_sample(rngm, rngm.randint(0, 79))
    img.save(f"{ROOT}/images/train/synth_{k:05}.jpg", quality=88)
    with open(f"{ROOT}/labels/train/synth_{k:05}.txt", "w") as fh:
        fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c, x, y, w, h in labels))
with open(f"{ROOT}/data.yaml", "w") as fh:
    fh.write(f"path: {ROOT}\ntrain: images/train\nval: images/val\nnc: {NC}\nnames: {NAMES}\n")

# como PRE-ROTULAR telas reais de graca (rode isto, corrija no LabelImg/Roboflow, mova pra labels/):
def prelabel(model, img_dir, out_dir):
    """Roda o F2 nas telas e escreve labels YOLO (voce so CORRIGE). Rotulo ~10x mais barato."""
    os.makedirs(out_dir, exist_ok=True)
    for p in sorted(glob.glob(f"{img_dir}/*.*")):
        r = model.predict(p, imgsz=IMGSZ, conf=0.35, verbose=False)[0]
        w, h = Image.open(p).size
        with open(f"{out_dir}/{os.path.splitext(os.path.basename(p))[0]}.txt", "w") as fh:
            for b in r.boxes:
                cx, cy, bw, bh = b.xywhn[0].tolist()
                fh.write(f"{int(b.cls[0])} {cx:.5f} {cy:.5f} {bw:.5f} {bh:.5f}\n")

# ---------- STEP 3: fine-tune (backbone congelado, LR baixo) ----------
box("[3/4] Fine-tune (backbone CONGELADO, LR baixo, poucas epocas)")
ft = YOLO(base_pt)
ft.train(data=f"{ROOT}/data.yaml", epochs=EPOCHS_FT, imgsz=IMGSZ, batch=32, freeze=FREEZE,
         lr0=LR0_FT, patience=10, device=0, project=ROOT, name="ft", verbose=True,
         seed=SEED, deterministic=True)

# ---------- STEP 4: BASE vs AJUSTADO no held-out (real) + export/publish se melhorou ----------
box("[4/4] BASE vs AJUSTADO no held-out + export")
m_base = base.val(data=f"{ROOT}/data.yaml", split="val", verbose=False)
m_ft = ft.val(data=f"{ROOT}/data.yaml", split="val", verbose=False)
b50, f50 = float(m_base.box.map50), float(m_ft.box.map50)
b95, f95 = float(m_base.box.map), float(m_ft.box.map)
base_maps = np.asarray(m_base.box.maps, dtype=float)
ft_maps = np.asarray(m_ft.box.maps, dtype=float)
if base_maps.shape != ft_maps.shape or base_maps.size != NC:
    raise RuntimeError("metricas por classe incompativeis com o schema de 54 classes")
per_class_delta = ft_maps - base_maps
per_class_floor_pass = bool(np.all(per_class_delta >= -0.02))
internal_gate_pass = (
    (not DEMO) and f50 >= b50 + 0.01 and f95 >= b95 and per_class_floor_pass
)
ganho = (f50 - b50) * 100
onnx = ft.export(format="onnx", imgsz=IMGSZ, opset=12)
onnx_path = Path(onnx)
if not onnx_path.is_file():
    raise RuntimeError("export ONNX ausente")
digest = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
candidate_dir = Path(ROOT) / "candidates" / digest
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_path = candidate_dir / "table_yolo11n_ft.onnx"
if candidate_path.exists():
    if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != digest:
        raise RuntimeError("colisao no diretorio imutavel do candidato")
else:
    shutil.copy2(onnx_path, candidate_path)
data_receipt_path = candidate_dir / 'real_data_receipt.json'
data_receipt_json = json.dumps(real_data_receipt, indent=2, sort_keys=True)
data_receipt_path.write_text(data_receipt_json, encoding='utf-8')
data_receipt_sha256 = hashlib.sha256(data_receipt_json.encode()).hexdigest()
manifest = {
    "schema_version": 1, "artifact_sha256": digest,
    "baseline_revision": HF_BASELINE_REVISION,
    "baseline_sha256": hashlib.sha256(Path(base_pt).read_bytes()).hexdigest(),
    "project_commit": PROJECT_COMMIT,
    "environment": {
        "python": platform.python_version(), "torch": torch.__version__,
        "cuda": torch.version.cuda, "ultralytics": ultralytics.__version__,
        "numpy": np.__version__, "pillow": PIL.__version__,
    },
    "seed": SEED, "groups": ([] if DEMO else srcs),
    "real_data_receipt_sha256": data_receipt_sha256,
    "dataset_manifest_receipt": (
        None if DEMO else governance_report.manifest_receipt.as_dict()
    ),
    "metrics": {"base_map50": b50, "candidate_map50": f50,
                "base_map50_95": b95, "candidate_map50_95": f95,
                "worst_per_class_delta": float(per_class_delta.min())},
    "per_class_floor_pass": per_class_floor_pass,
    "internal_gate_pass": internal_gate_pass,
    "exact_state_validated": False, "onnx_parity_validated": False,
    "promoted": False,
}
manifest_path = candidate_dir / "manifest.json"
manifest_json = json.dumps(manifest, indent=2, sort_keys=True)
if manifest_path.exists() and manifest_path.read_text(encoding="utf-8") != manifest_json:
    raise RuntimeError("manifesto imutavel ja existe com conteudo diferente")
if not manifest_path.exists():
    manifest_path.write_text(manifest_json, encoding="utf-8")
box("RESULTADO (fine-tune sim->real)", [
    f"held-out {'DEMO (dominio duro)' if DEMO else 'REAL'} - mAP50 BASE:    {b50 * 100:.1f}%",
    f"held-out {'DEMO (dominio duro)' if DEMO else 'REAL'} - mAP50 AJUSTADO: {f50 * 100:.1f}%",
    f"GANHO do fine-tune: {ganho:+.1f} pontos",
    f"pior delta por classe: {per_class_delta.min():+.3f}",
    ("stress-test sintetico: nao e evidencia real"
     if DEMO else f"gate interno agrupado: {'PASS' if internal_gate_pass else 'FAIL'}"),
])
print("candidato ONNX:", candidate_path)

if PUBLISH_ARTIFACTS:
    if not internal_gate_pass:
        raise RuntimeError("publicacao bloqueada: gate interno nao passou")
    from huggingface_hub import HfApi
    publish_token = (userdata.get('HF_TOKEN') if userdata is not None
                     else os.environ.get('HF_TOKEN'))
    os.environ.pop('HF_TOKEN', None)
    if not isinstance(publish_token, str) or not publish_token:
        raise RuntimeError('HF_TOKEN deve ser reapresentado somente para publicar.')
    api = HfApi(token=publish_token)
    publish_token = ''
    remote_dir = f"candidates/{digest}"
    publish_commits = {}
    for local in (candidate_path, manifest_path, data_receipt_path):
        commit = api.upload_file(path_or_fileobj=str(local),
                                 path_in_repo=f"{remote_dir}/{local.name}",
                                 repo_id=HF_REPO_ID, repo_type="model")
        if re.fullmatch(r'[0-9a-f]{40}', commit.oid) is None:
            raise RuntimeError('Hub nao retornou commit imutavel verificavel.')
        publish_commits[local.name] = commit.oid
    (candidate_dir / 'hf_publish_receipt.json').write_text(
        json.dumps({
            'remote_dir': remote_dir, 'commits': publish_commits,
            'artifact_sha256': digest, 'canonical_updated': False,
        }, indent=2, sort_keys=True), encoding='utf-8',
    )
    print("candidato publicado em", remote_dir, "; canonico inalterado")
    del api
else:
    print("nao publicado (padrao seguro; POKER_PUBLISH_ARTIFACTS != 1)")
print("\nFIM. Exact-state, calibracao e paridade ONNX ainda bloqueiam promocao.")